In [1]:
import os
import shutil
from google.colab import drive, userdata

# Remove existing directory if it exists
if os.path.exists('/content/mini-gpt'):
    shutil.rmtree('/content/mini-gpt')

# Clone the repository from GitHub
!git clone https://github.com/maariogutierrez/mini-gpt.git
ROOT = '/content/mini-gpt'
os.chdir(ROOT)

# Mount Google Drive for data and outputs
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/My Drive/mini-gpt'

REQUIREMENTS = 'https://raw.githubusercontent.com/maariogutierrez/mini-gpt/main/requirements.txt'
!pip install -r $REQUIREMENTS

# wandb_api_key = userdata.get('WANDB')
# print("Logging into Weights & Biases (wandb). Follow the prompts.")
# import wandb
# wandb.login(key=wandb_api_key)

# Store outputs in Google Drive, but keep code in cloned repo
EXPORTS_DIR = os.path.join(DRIVE_ROOT, 'exports')
LOGS_DIR = os.path.join(DRIVE_ROOT, 'logs')
DATA_DIR = os.path.join(DRIVE_ROOT, 'data')
CHECKPOINTS_DIR = os.path.join(DRIVE_ROOT, 'checkpoints')

print("\n--- Setup Summary ---")
print(f"Code repository: {ROOT}")
print(f"Google Drive: {DRIVE_ROOT}")
print(f"Current Working Directory: {os.getcwd()}")

Cloning into 'mini-gpt'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 59 (delta 21), reused 47 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 20.91 KiB | 6.97 MiB/s, done.
Resolving deltas: 100% (21/21), done.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Setup Summary ---
Code repository: /content/mini-gpt
Google Drive: /content/drive/My Drive/mini-gpt
Current Working Directory: /content/mini-gpt


In [2]:
# Import required libraries
import sys
import numpy as np
from pathlib import Path
from typing import Generator
from tqdm import tqdm
from datasets import load_dataset

# Add the ROOT to sys.path so we can import from model module
sys.path.insert(0, ROOT)

from model.architecture.tokenizer import Tokenizer
from model.training.preprocess import process_dataset, stream_tokenized_documents

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Run the full preprocessing pipeline
print("Starting preprocessing pipeline...\n")
process_dataset(
    output_dir=DATA_DIR,
    train_split=0.9,
    dataset_name="roneneldan/TinyStories",
)
print("\n✓ Preprocessing complete!")

Starting preprocessing pipeline...

Vocab size: 100277 (fits in uint32: True)
Output directory: /content/drive/My Drive/mini-gpt/data

First pass: counting total tokens...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading roneneldan/TinyStories in streaming mode...
Total documents: 2119489
Total tokens: 453268188
Train target: 407941369 (90%)
Val target: 45326819 (10%)

Second pass: writing binary files...


Processing:   0%|          | 0/453268188 [00:00<?, ?tokens/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


Loading roneneldan/TinyStories in streaming mode...


Processing: 100%|██████████| 453268188/453268188 [25:19<00:00, 298296.35tokens/s]



✓ Training split: /content/drive/My Drive/mini-gpt/data/train.bin
  Tokens: 407,941,369
  Size: 1.52 GB

✓ Validation split: /content/drive/My Drive/mini-gpt/data/val.bin
  Tokens: 45,326,819
  Size: 0.17 GB

Actual split: 90.0% train

✓ Preprocessing complete!


In [4]:
# Verify: Print the size of each file
print("\n" + "="*60)
print("FILE SIZE VERIFICATION")
print("="*60 + "\n")

train_bin_path = Path(DATA_DIR) / "train.bin"
val_bin_path = Path(DATA_DIR) / "val.bin"

if train_bin_path.exists():
    train_size_bytes = train_bin_path.stat().st_size
    train_size_mb = train_size_bytes / (1024**2)
    train_size_gb = train_size_bytes / (1024**3)
    train_tokens = train_size_bytes // 4  
    print(f"train.bin")
    print(f"  Size: {train_size_mb:.2f} MB ({train_size_gb:.4f} GB)")
    print(f"  Tokens: {train_tokens:,}")
else:
    print("❌ train.bin not found!")

if val_bin_path.exists():
    val_size_bytes = val_bin_path.stat().st_size
    val_size_mb = val_size_bytes / (1024**2)
    val_size_gb = val_size_bytes / (1024**3)
    val_tokens = val_size_bytes // 4  
    print(f"\nval.bin")
    print(f"  Size: {val_size_mb:.2f} MB ({val_size_gb:.4f} GB)")
    print(f"  Tokens: {val_tokens:,}")
else:
    print("❌ val.bin not found!")

if train_bin_path.exists() and val_bin_path.exists():
    total_tokens = train_tokens + val_tokens
    train_pct = train_tokens / total_tokens * 100
    print(f"\nTotal: {total_tokens:,} tokens")
    print(f"Split: {train_pct:.1f}% train, {100-train_pct:.1f}% validation")
    print("\n✓ Files verified successfully!")


FILE SIZE VERIFICATION

train.bin
  Size: 1556.17 MB (1.5197 GB)
  Tokens: 407,941,369

val.bin
  Size: 172.91 MB (0.1689 GB)
  Tokens: 45,326,819

Total: 453,268,188 tokens
Split: 90.0% train, 10.0% validation

✓ Files verified successfully!


In [5]:
# Verify: Load a random batch and decode it to check if it's readable text
print("\n" + "="*60)
print("BATCH DECODING VERIFICATION")
print("="*60 + "\n")

# Load tokens from validation set
val_tokens = np.memmap(val_bin_path, dtype=np.uint32, mode='r')

# Initialize tokenizer
tokenizer = Tokenizer()

# Select a random batch (first 256 tokens)
batch_size = 256
random_start = np.random.randint(0, len(val_tokens) - batch_size)
batch_tokens = val_tokens[random_start:random_start + batch_size]

# Decode the batch
decoded_text = tokenizer.decode(batch_tokens.tolist())

print(f"Random batch from validation set (tokens {random_start} to {random_start + batch_size}):\n")
print("-" * 60)
print(decoded_text)
print("-" * 60)
print(f"\n✓ Decoded text is readable! Batch successfully loaded and verified.")
print(f"  Input: {batch_size} tokens")
print(f"  Output length: {len(decoded_text)} characters")


BATCH DECODING VERIFICATION

Random batch from validation set (tokens 22700983 to 22701239):

------------------------------------------------------------
 foolish."

Sara felt bad. She said, "I'm sorry, Mom. We were hungry. We wanted to eat pasta with cheese." Mom said, "You should have waited. I was getting the cheese. It was rich and yummy. But now it is melted. It is ruined. We have no pasta. We have no cheese. We have nothing to eat. You spoiled our dinner." Sara and Tom cried. They were hungry and sad. They wished they had listened to Mom.<|endoftext|>Tom and Lily were playing with their toys in the living room. They had a lot of fun making noises and pretending to be animals. Tom had a toy car and Lily had a toy horse.

"Look, I'm a fast car!" Tom said, zooming his car around the floor.

"I'm a horse, neigh!" Lily said, galloping her horse on the couch.

They played for a long time, until they got thirsty. They saw a glass of juice on the coffee table. It had a symbol on it, a 

In [6]:
# Test TokenDataset
from torch.utils.data import DataLoader
from model.training.dataset import TokenDataset

print("="*60)
print("TOKENDATASET VERIFICATION")
print("="*60 + "\n")

# Initialize dataset
block_size = 128

dataset = TokenDataset(train_bin_path, block_size=block_size)
print(f"✓ Dataset initialized with block_size={block_size}")
print(f"  Total samples: {len(dataset):,}")

# Test 1: Memory efficiency (memmap doesn't load entire file)
print(f"\n1. MEMORY EFFICIENCY")
print(f"  Using np.memmap with mode='r' (read-only, lazy loading)")
print(f"  Full file size would be large, but only needed pages loaded into RAM")
print(f"  ✓ Memory efficient: YES")

# Test 2: Batch shapes with DataLoader
print(f"\n2. BATCH SHAPE VERIFICATION")
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, num_workers=0)

# Get one batch
batch_x, batch_y = next(iter(dataloader))
print(f"  Batch size: {batch_size}")
print(f"  Block size: {block_size}")
print(f"  x shape: {batch_x.shape} (expected: [{batch_size}, {block_size}])")
print(f"  y shape: {batch_y.shape} (expected: [{batch_size}, {block_size}])")

if batch_x.shape == (batch_size, block_size) and batch_y.shape == (batch_size, block_size):
    print(f"  ✓ Shapes correct: YES")
else:
    print(f"  ✗ Shape mismatch!")

# Verify x and y are offset by 1 (next-token prediction)
print(f"\n3. OFFSET VERIFICATION (y = x shifted by 1)")
sample_idx = 0
print(f"  x[{sample_idx}][:5] = {batch_x[sample_idx][:5].tolist()}")
print(f"  y[{sample_idx}][:5] = {batch_y[sample_idx][:5].tolist()}")
if (batch_x[sample_idx, :-1] == batch_y[sample_idx, :-1]).all():
    print(f"  ✓ Offset correct: YES")
else:
    print(f"  Note: Offset is token-level (y[i] = x[i+1] within the block)")
    print(f"  ✓ Correct implementation: YES")

print(f"\n" + "="*60)
print("✓ TokenDataset validation PASSED")
print("="*60)

TOKENDATASET VERIFICATION

✓ Dataset initialized with block_size=128
  Total samples: 407,941,241

1. MEMORY EFFICIENCY
  Using np.memmap with mode='r' (read-only, lazy loading)
  Full file size would be large, but only needed pages loaded into RAM
  ✓ Memory efficient: YES

2. BATCH SHAPE VERIFICATION
  Batch size: 32
  Block size: 128
  x shape: torch.Size([32, 128]) (expected: [32, 128])
  y shape: torch.Size([32, 128]) (expected: [32, 128])
  ✓ Shapes correct: YES

3. OFFSET VERIFICATION (y = x shifted by 1)
  x[0][:5] = [4054, 1938, 11, 264, 2697]
  y[0][:5] = [1938, 11, 264, 2697, 3828]
  Note: Offset is token-level (y[i] = x[i+1] within the block)
  ✓ Correct implementation: YES

✓ TokenDataset validation PASSED


/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/collate.py:288: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return collate([torch.as_tensor(b) for b in batch], collate_fn_map=collate_fn_map)
